# 08 · Human-on-the-loop · **Review tool calls**

**Idea:** The LLM *proposes* a tool call (name + arguments). Before it runs, a human can:

| Decision | Effect |
|---|---|
| ✅ **approve** | Run the tool exactly as proposed |
| ✏️ **edit** | Change the arguments (or tool) first, then run |
| ❌ **reject** | Don't run; the model is told why and can try something else |

LangChain ships this as a one-liner: **`HumanInTheLoopMiddleware`**. You choose *which* tools need review;
safe tools (read-only) run automatically.

## How human-in-the-loop works in LangGraph (read once)

1. A node calls **`interrupt(payload)`** → the graph **pauses** and saves its state to the **checkpointer**.
2. `graph.invoke(...)` returns right away with `"__interrupt__"` holding your payload (show it to the human).
3. Later, call `graph.invoke(Command(resume=<human answer>), config)` with the **same `thread_id`**.
4. The node **re-runs from its start**, and this time `interrupt(...)` *returns the human's answer*.

⚠️ Because the node restarts, put side-effects (sending email, charging cards) **after** the `interrupt()` call, never before.
A checkpointer + `thread_id` are mandatory — that's what remembers where the graph paused.

In [1]:
# --- Model setup -------------------------------------------------------------
# 1) Ollama must be running   ->  `ollama serve`  (the desktop app does this for you)
# 2) Pull the models once     ->  `ollama pull llama3.1`  and  `ollama pull qwen3:0.6b`
from langchain_ollama import ChatOllama

# llama3.1 (8B): our main "brain" - good at tool calling & structured output.
llm = ChatOllama(model="llama3.1", temperature=0)

# qwen3:0.6b: a tiny, very fast model - great for cheap jobs (classify, summarise, route).
# reasoning=False hides qwen3's <think> block so `.content` is just the answer.
small_llm = ChatOllama(model="qwen3:0.6b", temperature=0, reasoning=False)

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

@tool
def search_contacts(name: str) -> str:
    """Find a person's email address by name. Read-only and safe."""
    book = {"priya": "priya@example.com", "sam": "sam@example.com"}
    return book.get(name.lower(), "not found")

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email. Has real-world side effects!"""
    print(f"   📧 SENDING to={to} | subject={subject!r}")
    return f"Email sent to {to}"

In [12]:
def build_agent():
    return create_agent(
        llm,
        tools=[search_contacts, send_email],
        system_prompt="You are an assistant. Look up contacts with search_contacts, then use send_email. Don't halucinate, read the content correctly before reply",
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email": {"allowed_decisions": ["approve", "edit", "reject"]},  # needs review
                    "search_contacts": False,                                           # auto-run
                },
            )
        ],
        checkpointer=InMemorySaver(),       # required, same as any interrupt
    )

agent = build_agent()
ASK = {"messages": [{"role": "user", "content": "Email Priya to say the meeting moved to 4pm."}]}

def show_review_request(result):
    """Print what the human reviewer would see."""
    print("-----------------")
    print(result)
    print("--------XX---------")
    req = result["__interrupt__"][0].value["action_requests"][0]
    print("⏸️  Agent wants to call:", req["name"])
    print("   with args:", req["args"])

### Scenario 1 · ✅ Approve

In [10]:
cfg = {"configurable": {"thread_id": "review-approve"}}
r = agent.invoke(ASK, cfg)
show_review_request(r)

r = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), cfg)
print("🤖", r["messages"][-1].content)

-----------------
{'messages': [HumanMessage(content='Email Priya to say the meeting moved to 4pm.', additional_kwargs={}, response_metadata={}, id='f4f08f63-3e6b-4bee-8310-6f1d1558edb3'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-10-01T16:22:45.238589Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3339556208, 'load_duration': 24244166, 'prompt_eval_count': 249, 'prompt_eval_duration': 964377000, 'eval_count': 45, 'eval_duration': 2331330000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'}, id='lc_run--01a0f846-73d5-7b91-9e84-5f144249adb6-0', tool_calls=[{'name': 'search_contacts', 'args': {'name': 'Priya'}, 'id': '67548a5a-6ee6-40d7-b3be-d113a43d4278', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 249, 'output_tokens': 45, 'total_tokens': 294}), ToolMessage(content='priya@example.com', name='search_contacts', id='7fd4dbd5-4d18-414b-a838-7b3c78e11201', tool_c

### Scenario 2 · ✏️ Edit the arguments before running

In [14]:
cfg = {"configurable": {"thread_id": "review-edit"}}
r = agent.invoke(ASK, cfg)
show_review_request(r)

edited = {
    "name": "send_email",
    "args": {"to": "priya@example.com", "subject": "Meeting moved",
             "body": "Hi Priya, the meeting is now at 5pm . Thanks!"},   # human fixed the time
}
r = agent.invoke(Command(resume={"decisions": [{"type": "edit", "edited_action": edited}]}), cfg)
print("🤖 >>", r["messages"][-1].content)

-----------------
{'messages': [HumanMessage(content='Email Priya to say the meeting moved to 4pm.', additional_kwargs={}, response_metadata={}, id='3088038e-2170-401b-a40d-a762221b36c7'), AIMessage(content='} \n\nThen, the output of the search_contacts function is used to format the answer to the original user question.\n\nAssuming the output of the search_contacts function is {"type": "success", "result": {"email": "priya@example.com"}}, \n\nThe final response would be:\n\n{"name": "send_email", "parameters": {"to": "priya@example.com", "subject": "Meeting Time Change", "body": "The meeting has been moved to 4pm."}}', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-10-01T16:44:26.569511Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8323847625, 'load_duration': 25284583, 'prompt_eval_count': 262, 'prompt_eval_duration': 1283645000, 'eval_count': 118, 'eval_duration': 6999547000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider':

### Scenario 3 · ❌ Reject with a reason

In [15]:
cfg = {"configurable": {"thread_id": "review-reject"}}
r = agent.invoke(ASK, cfg)
show_review_request(r)

r = agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "message": "Don't email her - I'll tell her in person."}]}),
    cfg,
)
print("🤖", r["messages"][-1].content)       # model acknowledges & does NOT send

-----------------
{'messages': [HumanMessage(content='Email Priya to say the meeting moved to 4pm.', additional_kwargs={}, response_metadata={}, id='9b990f2c-732c-4346-9677-bb81b9d6a593'), AIMessage(content='} \n\nThen, the output of the search_contacts function is used to format the answer to the original user question.\n\nAssuming the output of the search_contacts function is {"type": "success", "result": {"email": "priya@example.com"}}, \n\nThe final response would be:\n\n{"name": "send_email", "parameters": {"to": "priya@example.com", "subject": "Meeting Time Change", "body": "The meeting has been moved to 4pm."}}', additional_kwargs={}, response_metadata={'model': 'llama3.1', 'created_at': '2026-10-01T16:46:07.947285Z', 'done': True, 'done_reason': 'stop', 'total_duration': 6334172541, 'load_duration': 11741375, 'prompt_eval_count': 262, 'prompt_eval_duration': 69785000, 'eval_count': 118, 'eval_duration': 6210162000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': '

### Notes
* If the model proposes **several** tool calls in one turn, `decisions` must be a list with one entry per call, in order.
* `"respond"` is a 4th decision type: the human answers *in place of* the tool (handy for "ask the user" tools).
* Want full control? Do the same by hand: a node between `agent` and `tools` that calls `interrupt()` with the
  pending `tool_calls` (notebook 05 shows the mechanics).